# 🚇 서울 지하철 혼잡도 분석 (정책입안자 대상)
**SCQA · 민토 피라미드 · 기초통계 · 시각화 → 제안 + Action Plan**

| SCQA | 내용 |
|---|---|
| **S 상황** | 서울 지하철은 하루 수백만 명의 핵심 교통수단 |
| **C 문제** | 코로나 후 수요가 빠르게 회복 → 혼잡 재심화 → 안전 위험 |
| **Q 질문** | 어느 호선·역·시간대가 혼잡한가? 정책으로 어떻게 풀까? |
| **A 답** | 혼잡은 **2호선·도심 업무지구·출퇴근 시간대**에 구조적 집중 → 집중 지점 우선 투자 + 수요 분산 |


## 0. 환경 설정 & 데이터 로드
> Colab이면 아래 폰트 설치 셀 먼저 1회 실행

In [ ]:
# (Colab 최초 1회) 한글 폰트 설치
!sudo apt-get install -y -qq fonts-nanum >/dev/null 2>&1
!fc-cache -fv >/dev/null 2>&1
!rm -rf ~/.cache/matplotlib

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumBarunGothic'   # 로컬(mac)이면 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 데이터 로드 (★ 반드시 cp949 — utf-8로 읽으면 깨짐)
FILE = '서울시 지하철 호선별 역별 시간대별 승하차 인원 정보.csv'
df = pd.read_csv(FILE, encoding='cp949')
print('원본:', df.shape)

## 1. 데이터 정제 (총승하차 파생 + 중복 제거)

In [ ]:
승차컬 = [c for c in df.columns if c.endswith('승차인원')]
하차컬 = [c for c in df.columns if c.endswith('하차인원')]
df['총승차']  = df[승차컬].sum(axis=1)
df['총하차']  = df[하차컬].sum(axis=1)
df['총승하차'] = df['총승차'] + df['총하차']
df = df.drop_duplicates(subset=['사용월','호선명','지하철역'])  # 2026-03 중복 제거
df['연도'] = df['사용월'] // 100

LATEST = df['사용월'].max()            # 가장 최근 월 = 혼잡 단면 분석 기준
cur = df[df['사용월'] == LATEST].copy()
print('정제 후:', df.shape, '| 기준월:', LATEST, f'({len(cur)}개 역)')

## 2. 근거 1 — 연도별 이용객 추이 (코로나 충격/회복)
**기초통계(합계) + 시계열.** 코로나(2020) 충격과 회복 흐름 확인.

In [ ]:
yr = df.groupby('연도')['총승하차'].sum() / 1e8   # 억 명
print(yr.round(1))
print(f"2020 코로나: 2019 대비 {(yr[2020]/yr[2019]-1)*100:+.1f}%")

fig, ax = plt.subplots(figsize=(9,4))
yr.plot(marker='o', ax=ax, color='#E63946')
ax.set_title('연도별 지하철 총 이용객 추이'); ax.set_ylabel('억 명'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3. 근거 2 — 호선별 이용객 차이 (ANOVA)
**ANOVA**로 호선 간 역당 이용객 차이가 통계적으로 유의한지 검정.

In [ ]:
line = (cur.groupby('호선명')['총승하차']
          .agg(총합='sum', 역당평균='mean', 역수='count')
          .sort_values('총합', ascending=False))
print(line.head(8).round(0))

groups = [g['총승하차'].values for _, g in cur.groupby('호선명') if len(g) >= 5]
F, p = stats.f_oneway(*groups)
print(f"\nANOVA  F={F:.1f},  p={p:.2e}  → 호선 간 차이 {'유의' if p<0.05 else '무의미'}")

fig, ax = plt.subplots(figsize=(9,4))
line.head(8)['총합'].div(1e6)[::-1].plot.barh(ax=ax, color='#457B9D')
ax.set_title('호선별 총 이용객 Top8 (기준월)'); ax.set_xlabel('백만 명')
plt.tight_layout(); plt.show()

## 4. 근거 3 — 시간대 출퇴근 피크 & 유입/유출형 역
출근(아침 하차 피크)·퇴근(저녁 승차 피크) 패턴 + 업무지구(유입형) vs 주거지(유출형) 구분.

In [ ]:
승t = cur[승차컬].sum(); 하t = cur[하차컬].sum()
print('승차 피크:', 승t.idxmax(), f'({승t.max()/1e6:.2f}백만)')
print('하차 피크:', 하t.idxmax(), f'({하t.max()/1e6:.2f}백만)')

# 아침(07~09시) 하차/승차 비 → 높으면 업무지구(유입형)
am승 = ['07시-08시 승차인원','08시-09시 승차인원']
am하 = ['07시-08시 하차인원','08시-09시 하차인원']
cur['아침승'] = cur[am승].sum(axis=1); cur['아침하'] = cur[am하].sum(axis=1)
cur['유입비'] = cur['아침하'] / (cur['아침승'] + 1)
big = cur[cur['아침승'] > 5000]
print('\n[유입형=업무지구]'); print(big.nlargest(5,'유입비')[['지하철역','호선명','유입비']].to_string(index=False))
print('\n[유출형=주거지]');  print(big.nsmallest(5,'유입비')[['지하철역','호선명','유입비']].to_string(index=False))

fig, ax = plt.subplots(figsize=(11,4)); x = range(len(승차컬))
ax.plot(x, 승t.values/1e6, marker='o', label='승차', color='#457B9D')
ax.plot(x, 하t.values/1e6, marker='s', label='하차', color='#E63946')
ax.set_xticks(x); ax.set_xticklabels([c.split('시-')[0] for c in 승차컬], rotation=90, fontsize=7)
ax.set_title('시간대별 승하차 (기준월)'); ax.set_ylabel('백만'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5. 근거 4 — 역별 혼잡 Top & 집중도
가장 붐비는 역 + 혼잡이 소수 역에 얼마나 쏠리는지(집중도).

In [ ]:
top = cur.nlargest(10,'총승하차')[['지하철역','호선명','총승하차']]
print(top.to_string(index=False))

s = cur['총승하차'].sort_values(ascending=False).values
n10 = max(1, len(s)//10)
print(f"\n상위 10%({n10}역)이 전체의 {s[:n10].sum()/s.sum()*100:.1f}% 차지 (집중도)")

fig, ax = plt.subplots(figsize=(9,4))
top[::-1].plot.barh(x='지하철역', y='총승하차', ax=ax, color='#2A9D8F', legend=False)
ax.set_title('역별 이용객 Top10 (기준월)'); plt.tight_layout(); plt.show()

## 6. 🏛️ 핵심 결론 → 제안 → Action Plan (정책입안자)

> **핵심 결론**: 서울 지하철 혼잡은 **2호선·도심 업무지구·출퇴근 시간대**에 구조적으로 집중되며, 코로나 후 수요가 빠르게 회복(2024 거의 회복)돼 **재혼잡**이 진행 중이다.

| 진단(근거) | 정책 제안 | Action Plan (정책수단 / 주체 / 시점 / KPI) |
|---|---|---|
| 근거1 재혼잡 | 선제적 혼잡관리 | 단기: 혼잡 모니터링·정보공개 / 서울시·교통공사 / *안전사고 0건* |
| 근거2 2호선 집중 | 선택과 집중 예산 | 중기: 2호선·집중역 시설·동선 개선 우선 / 서울시 / *집중역 혼잡도↓* |
| 근거3 피크 집중 | 수요 분산 제도 | 중기: 시차출퇴근·유연근무 인센티브+증차 / 국토부·기업 / *피크 분산율* |
| 근거4 도심 집중 | 광역·직주근접 | 장기: 광역교통·도시계획 연계 / 국토부·서울시 / *도심 통행집중도↓* |

*분석 단위: 역×월 / 기준월 단면 + 2015~2026 추이*
